In [1]:
import pandas as pd
import requests
import datetime
from alto_connect import AltoConnect

In [2]:
#pip install alto_connect

# Functions to export and to do data processing

## You can find all the details regarding the class Alto Connect here: https://alto-connect-docs.intramundi.com/alto_connect.html

In [3]:
def report(*portfolios,module,view,date=datetime.datetime.today(),to_excel=False):
    #function to return views (table) of multiple funds of a modul 
    #This will return a dictionnary of dataframes
    #If option "to_excel" is True then it will produce an excel file of all the tables
    
    alto_connect = AltoConnect()
    
    dataframes={}
    
    path=r'P:\Amundi_Dublin\Risk\Inv_Risk\Daily monitoring\ '
    
    for portfolio in portfolios:
    
        alto_connect.load_portfolio_list(ptflist=portfolio,module_name=module,date=date)
        dataframes[portfolio] = alto_connect.get_view(module_name = module, viewname= view)
    
    if to_excel:
        
        for portfolio in portfolios:   
            dataframes[portfolio].to_excel(path + portfolio+'_'+view+'.xlsx')

    return dataframes

In [4]:
def calculate_high_impact_weight(dataframe):
    
    #Function to calculate the weight of high impact sector, related to the ESG check on Net Zero Guidelines
    
    Cleaned_data=dataframe.copy()
    
    Cleaned_data['Wght% (REF)']=Cleaned_data['Wght% (REF)'].str.rstrip("%").astype("float") / 100
    Cleaned_data['Wght% (PTF)']=Cleaned_data['Wght% (PTF)'].str.rstrip("%").astype("float") / 100
    
    High_Impact=Cleaned_data[Cleaned_data['High Impact Climate Sector']=='Yes']
    weights=High_Impact[['Wght% (PTF)','Wght% (REF)']].sum()
    
    return weights


In [5]:
class ESG_checks:
    
    # This class allows to retrieve the specific tables of each modules regarding the ESG Checks, done weekly
    # It is mainly based on a Quant python class Alto Connect, 
    # which allow the user to connect to the HTML links of the table in Alto
    
    def __init__(self):
            
        self.alto=AltoConnect()
        self.path=r'P:\Amundi_Dublin\Risk\Inv_Risk\Daily monitoring\ '
        
    def report(self,*portfolios,module,view,date=datetime.datetime.today(),url=False,to_excel=False):
        #Main method that will get specific table on a specific module, with an Excel Export in option
        #Sometimes, it can be difficult to retrieve the data, so a parameter link has been set in case the user
        # wants to put the link of the table directly
        #The function will return a dictionnary of dataframes, thus all the tables for the given portfolios
        
        dataframes={}
        
        
        if not url:
        #if no url has been entered, function will use Alto Connect properties directly
        
            if to_excel:
            #Will make an excel export for all the portfolios
            
                for portfolio in portfolios:
                # Will create a dictionnary, with each associated key the table looked of

                    self.alto.load_portfolio_list(ptflist=portfolio,module_name=module,date=date)
                    dataframes[portfolio] = self.alto.get_view(module_name = module, viewname= view).to_excel(self.path+portfolio+'_'+view+'.xlsx')            

            for portfolio in portfolios:
                
                #if not export option selected, will only return the dictionnary of tables

                self.alto.load_portfolio_list(ptflist=portfolio,module_name=module,date=date)
                dataframes[portfolio] = self.alto.get_view(module_name = module, viewname= view)
                
                
        else:
            
            if to_excel:
            
                for portfolio in portfolios:
                
                    self.alto.load_portfolio_list(ptflist=portfolio,module_name=module,date=date)
                    dataframes[portfolio]=pd.read_html(url)[0].to_excel(self.path+portfolio+'_'+view+'.xlsx')
                    
            for portfolio in portfolios:
                #Will return the output of the table URL
                
                self.alto.load_portfolio_list(ptflist=portfolio,module_name=module,date=date)
                dataframes[portfolio]=pd.read_html(url)[0]
                    
                
        return dataframes
   
        
    def SRI_Rating_check(self,*portfolios,date=datetime.datetime.today(),to_excel=False):
        #Method to return table regarding rating based on MSCI above CCC
        dataframes=self.report(*portfolios,module = 'globalAnalysis',view='Ratings based on MSCI - Above CCC',date=date,to_excel=to_excel)
            
        return dataframes
    
    def SRI_Average_check(self,*portfolios,date=datetime.datetime.today(),to_excel=False):
        #Method to give ESG average rating
    
        dataframes=self.report(*portfolios,module = 'globalAnalysis',view='ESG Average Ratings',date=date,to_excel=to_excel)
            
        return dataframes
    
    def SRI_check_top80(self,*portfolios,date=datetime.datetime.today(),to_excel=False):
        
        #Method to give the ESG average rating of the fund agains the top 80% ratings of the benchmark 
        
        
        dataframes_average=self.report(*portfolios,module = 'globalAnalysis',view='ESG Average Ratings',date=date,to_excel=False)
        dataframes_top80=self.report(*portfolios,module = 'globalAnalysis',view='REF top 80% ESG Score - New Methodo',date=date,to_excel=False)
        
        dataframes={}
        for portfolio in portfolios:
            
            dataframes[portfolio]=pd.concat([dataframes_average[portfolio],dataframes_top80[portfolio]])    
               
            if to_excel:
                
                dataframes[portfolio].to_excel(self.path+portfolio+' ESG Top 80.xlsx')
                
        return dataframes
        
    
    def ESG_BtB_check(self,*portfolios,date=datetime.datetime.today(),to_excel=False):
        
        #method that return the Beat The Bench Table
    
        dataframes=self.report(*portfolios,module = 'globalAnalysis',view='BtB_ESG_conditions',date=date,to_excel=to_excel)

            
        return dataframes    
    
    def High_Impact_climate_Sector_check(self,*portfolios,date=datetime.datetime.today(),to_excel=False):
        
        #Method that gives the whole inventory of the fund, and the high impact climate sectors weights
        dataframes=self.report(*portfolios,module = 'globalAnalysis',view='Inventory',date=date,to_excel=False)
        
        for portfolio in portfolios:
            dataframes[portfolio].columns=dataframes[portfolio].columns.get_level_values(1)
            
            if to_excel:
            
                dataframes[portfolio].to_excel(self.path + portfolio+' Inventory Alignment on Net Zero Guidelines.xlsx')
        
        return dataframes

# Sabadell Commitment Checks

In [6]:
Dataframes=report('6694','5438',module= 'commitmentLeverage',view='ESMA by Instrument Group and type',to_excel=True)

C:\Users\selvam\AppData\Roaming\Python\Python39\site-packages\pip_system_certs\wrapt_requests.py:71: UserWarning: Failed to patch SSL settings for unverified requests (unsupported version of urllib3?)
This may lead to errors when urllib3 tries to modify verify_mode.
Please report an issue at https://gitlab.com/alelec/pip-system-certs with your
python version included in the description

  warnings.warn(


2024-11-27 16:08:16 INFO [Filename __init__.py:142 in function load_portfolio_list] http://webalto.sits.credit-agricole.fr/altoservices/selvam/alto/?action=loadPortfolioList&module=commitmentLeverage&ptfCodeList=6694&cleardataset=true&reloadNotCleared=true&date=27%2F11%2F2024&securemode=true
2024-11-27 16:08:16 INFO [Filename __init__.py:255 in function get_view] http://webalto.sits.credit-agricole.fr/altoservices/selvam/alto/?module=commitmentLeverage&view=ESMA+by+Instrument+Group+and+type
2024-11-27 16:08:20 INFO [Filename __init__.py:142 in function load_portfolio_list] http://webalto.sits.credit-agricole.fr/altoservices/selvam/alto/?action=loadPortfolioList&module=commitmentLeverage&ptfCodeList=5438&cleardataset=true&reloadNotCleared=true&date=27%2F11%2F2024&securemode=true
2024-11-27 16:08:20 INFO [Filename __init__.py:255 in function get_view] http://webalto.sits.credit-agricole.fr/altoservices/selvam/alto/?module=commitmentLeverage&view=ESMA+by+Instrument+Group+and+type


In [7]:
Dataframes['5438']

,ESMA leverage weight,ESMA leverage,Addon coef,CoverRules_CoverageLevel,Coverage Level%,ISIN,Market exposure,Mkt.Expo (PTF),Mkt.Expo% (PTF),Quantity,...,Underlying Inst quote Future,Inst quote Future,Long/short,Rate risk maturity date,Funds contribution,Pocket exposure,Asset Class Standard,Asset Class CSSF Leverage,Asset class CSSF leverage Ordered,Instrument group CSSF
5438,3.90 %,5 286 185,2.00 %,115 052,0.08%,NaN,135 598 593,135 584 634,99.99 %,NaN,...,0,0,NaN,23/01/25,NaN,0,NaN,NaN,NaN,NaN
Forex,3.90 %,5 286 185,2.00 %,115 052,0.08%,NaN,-9 328,-23 287,-0.02 %,NaN,...,0,0,NaN,23/01/25,NaN,0,Money Market,Forex,Foreign exchange,Forex
Cash,0.00 %,0,NaN,0,0%,NaN,1 754 314,1 754 314,1.29 %,NaN,...,0,0,NaN,NaN,0,0,Money Market,Money Market,Fixed Income/interest rate,Cash
Equities,0.00 %,0,NaN,0,0%,NaN,133 853 607,133 853 607,98.71 %,NaN,...,0,0,NaN,NaN,0,0,Equities,Equities,Equities,Equities


In [8]:
Dataframes['6694']

,ESMA leverage weight,ESMA leverage,Addon coef,CoverRules_CoverageLevel,Coverage Level%,ISIN,Market exposure,Mkt.Expo (PTF),Mkt.Expo% (PTF),Quantity,...,Underlying Inst quote Future,Inst quote Future,Long/short,Rate risk maturity date,Funds contribution,Pocket exposure,Asset Class Standard,Asset Class CSSF Leverage,Asset class CSSF leverage Ordered,Instrument group CSSF
6694,0.00 %,0,NaN,0,0%,NaN,169 089 447,169 089 447,100.00 %,NaN,...,0,0,NaN,NaN,0,0,NaN,NaN,NaN,NaN
Cash,0.00 %,0,NaN,0,0%,NaN,2 569 389,2 569 389,1.52 %,NaN,...,0,0,NaN,NaN,0,0,Money Market,Money Market,Fixed Income/interest rate,Cash
Equities,0.00 %,0,NaN,0,0%,NaN,166 520 058,166 520 058,98.48 %,NaN,...,0,0,Long,NaN,0,0,Equities,Equities,Equities,Equities


In [9]:
url="http://webalto.sits.credit-agricole.fr/altoservices/selvam/alto/?module=APKgui&view=Holdings"
test=report("PERPVITPC",module="APKgui",view="Holdings",url=url)

TypeError: report() got an unexpected keyword argument 'url'

In [13]:
test['PERPVITPC']

,0,1,2,3,4,5,6,7,8,9,...,52,53,54,55,56,57,58,59,60,61
0,+,Show Detail,Position Type Code,Strategy Code,Instrument Number,ISIN,Quantity,Portfolio Currency,Valuation (Ptf),Valuation (Inst),...,Quote Close,Clean Value / NAV Front (%),Value / NAV Off (%),Clean Value/ NAV Tech (%),Value / NAV Tech (%),Clean Value / NAV Off (%),Value / NAV Front (%),Cntrpty Cod,Cntrpty Name,Fiscality
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,PERPVITPC | PERP VITALITE - POCHE CROSSOVER,NaN,NaN,NaN,NaN,NaN,NaN,EUR,5 979 044.79 EUR,5 979 044.79 EUR,...,NaN,94.291784%,597904479%,94.291784%,100%,563774800%,100%,NaN,NaN,NaN
3,BONDS,NaN,NOR,CORE,NaN,NaN,NaN,EUR,5 724 294.79 EUR,5 724 294.79 EUR,...,NaN,94.291784%,572429479%,94.291784%,95.739286%,563774800%,95.739286%,NaN,NaN,NaN
4,AMPLIFON SPA 1.125% Feb27,NaN,NOR,CORE,7315210,XS2116503546,1 100 000.00 N,EUR,1 062 920.08 EUR,1 062 920.08 EUR,...,95.75,17.61569%,106292008%,17.61569%,17.777423%,105325000%,17.777423%,NaN,NaN,NaN
5,IMCD NV 2.5% Mar25,NaN,NOR,CORE,5721799,XS1791415828,4 600 000.00 N,EUR,4 661 374.71 EUR,4 661 374.71 EUR,...,99.663,76.676094%,466137471%,76.676094%,77.961863%,458449800%,77.961863%,NaN,NaN,NaN
6,CURRENCY,NaN,VAL,NaN,251355,NaN,254 750.00 EUR,EUR,254 750.00 EUR,254 750.00 EUR,...,NaN,NaN,25475000%,NaN,4.260714%,NaN,4.260714%,NaN,NaN,NaN
7,EURO,NaN,VAL,NaN,251355,NaN,254 750.00 EUR,EUR,254 750.00 EUR,254 750.00 EUR,...,NaN,NaN,25475000%,NaN,4.260714%,NaN,4.260714%,NaN,NaN,NaN


In [10]:
def report(*portfolios,module,view,date=datetime.datetime.today(),url=False,to_excel=False):
    #Main method that will get specific table on a specific module, with an Excel Export in option
    #Sometimes, it can be difficult to retrieve the data, so a parameter link has been set in case the user
    # wants to put the link of the table directly
    #The function will return a dictionnary of dataframes, thus all the tables for the given portfolios

    
    alto_connect = AltoConnect()
    dataframes={}
    path=r'P:\Amundi_Dublin\Risk\Inv_Risk\Daily monitoring\ '


    if not url:
    #if no url has been entered, function will use Alto Connect properties directly

        for portfolio in portfolios:

            #if not export option selected, will only return the dictionnary of tables

            alto_connect.load_portfolio_list(ptflist=portfolio,module_name=module,date=date)
            dataframes[portfolio] = alto_connect.get_view(module_name = module, viewname= view)

    else:

        for portfolio in portfolios:
                #Will return the output of the table URL

            alto_connect.load_portfolio_list(ptflist=portfolio,module_name=module,date=date)
            dataframes[portfolio]=pd.read_html(url)[0]
                
    if to_excel:
        
        for portfolio in portfolios:
            
            dataframes[portfolio].to_excel(path+portfolio+'_'+view+'.xlsx')            


    return dataframes

In [ ]:
url='http://webalto.sits.credit-agricole.fr/altoservices/selvam/alto/?module=globalAnalysis&view=Derivatives+Holdings'
test=report('E143','E142','E119',module='globalAnalysis',view='Holding',url=url)

C:\Users\selvam\AppData\Roaming\Python\Python39\site-packages\pip_system_certs\wrapt_requests.py:71: UserWarning: Failed to patch SSL settings for unverified requests (unsupported version of urllib3?)
This may lead to errors when urllib3 tries to modify verify_mode.
Please report an issue at https://gitlab.com/alelec/pip-system-certs with your
python version included in the description

  warnings.warn(


2024-10-14 16:14:27 INFO [Filename __init__.py:142 in function load_portfolio_list] http://webalto.sits.credit-agricole.fr/altoservices/selvam/alto/?action=loadPortfolioList&module=globalAnalysis&ptfCodeList=E143&cleardataset=true&reloadNotCleared=true&date=14%2F10%2F2024&securemode=true
2024-10-14 16:14:44 INFO [Filename __init__.py:142 in function load_portfolio_list] http://webalto.sits.credit-agricole.fr/altoservices/selvam/alto/?action=loadPortfolioList&module=globalAnalysis&ptfCodeList=E142&cleardataset=true&reloadNotCleared=true&date=14%2F10%2F2024&securemode=true
2024-10-14 16:14:54 INFO [Filename __init__.py:142 in function load_portfolio_list] http://webalto.sits.credit-agricole.fr/altoservices/selvam/alto/?action=loadPortfolioList&module=globalAnalysis&ptfCodeList=E119&cleardataset=true&reloadNotCleared=true&date=14%2F10%2F2024&securemode=true


In [ ]:
test['E119']